In [54]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
import numpy as np

In [55]:
class QuadState(TypedDict):
    coeff : list
    equation : str
    D : float
    root : list

In [56]:
def get_equation(state:QuadState):
    result = f"{state["coeff"][0]}x**2 + {state['coeff'][1]}x + {state['coeff'][2]}"
    return {"equation":result}

In [57]:
def get_d(state:QuadState):
    D = np.square(state['coeff'][1]) - 4*state["coeff"][0]*state['coeff'][2]
    return {"D":D}

In [58]:
def real_roots(state:QuadState):
    D = state.get("D", 0)
    r1 = (-state['coeff'][1] + np.sqrt(D))/2*state["coeff"][0]
    r2 = (-state['coeff'][1] - np.sqrt(D))/2*state["coeff"][0]
    return {"root": ["real_roots:", r1, r2]}

def single_root(state:QuadState):
    r1 = (-state['coeff'][1])/2*state["coeff"][0]
    return {'root' : ["single_root:" ,r1]}

def no_root(state:QuadState):
    D = state.get("D", 0)
    r1 = f"({-state['coeff'][1]} + {np.sqrt(D)}i)/{2*state["coeff"][0]}"
    r2 = f"({-state['coeff'][1]} - {np.sqrt(D)}i)/{2*state["coeff"][0]}"

    return {"root": ["imaginary_roots:", r1, r2]}



In [59]:
def check(state:QuadState)->Literal["real_roots", "single_root", "no_root"]:
    D = state.get("D", 0)
    if D>0:
        return "real_roots"
    elif D<0:
        return "no_root"
    else:
        return "single_root"

In [60]:
graph = StateGraph(QuadState)

graph.add_node("get_equation", get_equation)
graph.add_node("get_d", get_d)
graph.add_node("real_roots", real_roots)
graph.add_node("no_root", no_root)
graph.add_node("single_root", single_root)
# graph.add_node("check", check)



graph.add_edge(START, "get_equation")
graph.add_edge("get_equation", "get_d")
graph.add_conditional_edges("get_d", check)
graph.add_edge("real_roots", END)
graph.add_edge("no_root", END)
graph.add_edge("single_root", END)






In [61]:
workflow = graph.compile()

In [63]:
initial_state = {"coeff" : [10,4,1]}
final_state=  workflow.invoke(initial_state)
print(final_state)

{'coeff': [10, 4, 1], 'equation': '10x**2 + 4x + 1', 'D': np.int64(-24), 'root': ['imaginary_roots:', '(-4 + nani)/20', '(-4 - nani)/20']}


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2200\1762894740.py:13: RuntimeWarning: invalid value encountered in sqrt
  r1 = f"({-state['coeff'][1]} + {np.sqrt(D)}i)/{2*state["coeff"][0]}"
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2200\1762894740.py:14: RuntimeWarning: invalid value encountered in sqrt
  r2 = f"({-state['coeff'][1]} - {np.sqrt(D)}i)/{2*state["coeff"][0]}"
